**Data source.** I use the **Bank of England’s official yield-curve archive**, specifically the **Daily overnight index swap (OIS) curve: archive data** downloaded from the Bank’s yield-curves statistics page. The OIS curve is a sterling risk-free term structure derived from overnight index swap rates and is a reputable public source for UK interest-rate modelling. The Bank states that the published **spot and forward yields are continuously compounded and quoted on an annual basis**. For non-gilt instruments such as OIS, the Bank uses the **Actual/365** day-count convention. ([Bank of England][1])

[1]: https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/frequently-asked-questions-yield-curves.pdf?utm_source=chatgpt.com "Yield Curves - Frequently Asked Questions"


In [1]:
import pandas as pd
import numpy as np

WORKBOOK = "OIS daily data_2025 to present.xlsx"
SHEET = "4. spot curve"

raw = pd.read_excel(WORKBOOK, sheet_name=SHEET, header=None)

maturities = raw.iloc[3, 1:].astype(float).to_numpy()

data = raw.iloc[4:, :].copy()
data.columns = ["date"] + list(maturities)
data["date"] = pd.to_datetime(data["date"])

# keep January 2026 only
jan = data[(data["date"] >= "2026-01-01") & (data["date"] <= "2026-01-31")].copy()

# save long/tidy version
long = jan.melt(id_vars="date", var_name="t", value_name="zero_rate_pct").copy()
long["t"] = long["t"].astype(float)
long["zero_rate_cc"] = long["zero_rate_pct"] / 100.0


long = long[["date", "t", "zero_rate_cc"]].copy()
long.to_parquet("ois_spot_curve_jan_2026_long.parquet", index=False)

In [2]:
import pandas as pd
import numpy as np

WORKBOOK = "OIS daily data_2025 to present.xlsx"
SHEET = "4. spot curve"
VALUATION_DATE = "2026-01-30"

dt = 0.5
T = 10.0
grid = np.linspace(dt, T, int(T / dt))

raw = pd.read_excel(WORKBOOK, sheet_name=SHEET, header=None)

maturities = raw.iloc[3, 1:].astype(float).to_numpy()

data = raw.iloc[4:, :].copy()
data.columns = ["date"] + list(maturities)
data["date"] = pd.to_datetime(data["date"])

curve_date = data.loc[data["date"] <= VALUATION_DATE, "date"].max()

curve_day = (
    data.loc[data["date"] == curve_date]
    .drop(columns="date")
    .iloc[0]
    .astype(float)
)

t_obs = curve_day.index.to_numpy(dtype=float)
z_obs = curve_day.to_numpy(dtype=float) / 100.0

P_obs = np.exp(-z_obs * t_obs)
logP_grid = np.interp(grid, t_obs, np.log(P_obs))
P_market = np.exp(logP_grid)
z_grid = -np.log(P_market) / grid

market_curve = pd.DataFrame({
    "date": curve_date,
    "t": grid,
    "zero_rate_cc": z_grid,
    "zero_price": P_market,
})

print(market_curve)

         date     t  zero_rate_cc  zero_price
0  2026-01-30   0.5      0.036040    0.982142
1  2026-01-30   1.0      0.034917    0.965686
2  2026-01-30   1.5      0.034730    0.949239
3  2026-01-30   2.0      0.034926    0.932532
4  2026-01-30   2.5      0.035263    0.915617
5  2026-01-30   3.0      0.035650    0.898571
6  2026-01-30   3.5      0.036053    0.881451
7  2026-01-30   4.0      0.036459    0.864300
8  2026-01-30   4.5      0.036865    0.847137
9  2026-01-30   5.0      0.037273    0.829972
10 2026-01-30   5.5      0.037682    0.812814
11 2026-01-30   6.0      0.038094    0.795674
12 2026-01-30   6.5      0.038508    0.778566
13 2026-01-30   7.0      0.038922    0.761508
14 2026-01-30   7.5      0.039335    0.744520
15 2026-01-30   8.0      0.039746    0.727625
16 2026-01-30   8.5      0.040153    0.710845
17 2026-01-30   9.0      0.040554    0.694206
18 2026-01-30   9.5      0.040948    0.677731
19 2026-01-30  10.0      0.041333    0.661444


In [3]:
TR_PROXY = 0.5
HISTORY_START = pd.Timestamp("2025-01-01")

hist = (
    data.loc[(data["date"] >= HISTORY_START) & (data["date"] <= VALUATION_DATE), ["date", TR_PROXY]]
    .dropna()
    .sort_values("date")
)

r = hist[TR_PROXY].astype(float).to_numpy() / 100.0

sigma_hl = np.std(np.diff(r), ddof=1) * np.sqrt(252)
sigma_bdt = np.std(np.diff(np.log(r)), ddof=1) * np.sqrt(252)

print("sigma_hl =", sigma_hl)
print("sigma_bdt =", sigma_bdt)

sigma_hl = 0.002534732493927997
sigma_bdt = 0.06310160515041566
